# Fit the final model with the selected hyper-parameters

Uses the hyper-parameters chosen by the Optuna study (`demo_xgb_optuna_gpu.ipynb`), or a dict you set by hand,
to fit a Dask GPU XGBoost model, evaluate it and save it.

Two ways to fit:

| `FIT_MODE` | Training data | Number of trees | When to use |
|---|---|---|---|
| `"early_stopping"` | train | early stopping (50 rounds) on valid | reproduces the tuned model; valid stays a clean hold-out |
| `"train_plus_valid"` | train + valid | fixed: `(best_iteration + 1) × ROUND_SCALE` from the tuning trial | uses all labelled data for the final model; evaluate on a separate **test** set |

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # repo root, if the package is not pip-installed

from xgb_optuna_gpu import (
    build_dmatrices, decile_table, get_loss, load_dask_data, load_dask_split, load_tuned_params, make_client,
    make_synthetic_data, predict, regression_metrics, save_model_bundle, to_numpy, train_xgb_model,
)

## 1. Configuration

In [ ]:
USE_GPU = True

# Data. Leave TRAIN_PATH as None to use the synthetic demo data.
TRAIN_PATH = None            # e.g. "/Volumes/<catalog>/<schema>/<volume>/data/train"
VALID_PATH = None
TEST_PATH = None             # optional hold-out for the final evaluation (strongly advised with "train_plus_valid")
TARGET = "y"
FEATURES = None              # None -> every column except TARGET
FILE_FORMAT = "parquet"

# Where the hyper-parameters come from
PARAM_SOURCE = "study"       # "study" or "manual"
STUDY_DIR = Path("optuna_runs/xgb_tweedie")   # <output_dir>/<study_name> used for tuning (can be on a volume)
TRIAL = None                 # None -> best trial of the study; or a trial number, e.g. 37
MAX_OVERFIT = 0.10           # rule used to pick the best trial (same as in tuning)

# Used when PARAM_SOURCE == "manual" (or when the demo study does not exist)
LOSS = "tweedie"             # "mse", "huber" or "tweedie"
MANUAL_PARAMS = {
    "max_depth": 6, "learning_rate": 0.05, "gamma": 1e-3, "min_child_weight": 1.0,
    "max_delta_step": 0.7, "reg_alpha": 1e-3, "reg_lambda": 1.0,
    "subsample": 0.8, "colsample_bytree": 0.8, "max_bin": 256,
    "tweedie_variance_power": 1.5,   # tweedie only; for huber set "huber_slope"
}
MANUAL_NUM_ROUNDS = 1000     # trees for "train_plus_valid" with manual params

FIT_MODE = "early_stopping"  # "early_stopping" or "train_plus_valid"
NUM_BOOST_ROUND = 5000       # upper bound for "early_stopping"
EARLY_STOPPING_ROUNDS = 50
ROUND_SCALE = 1.0            # "train_plus_valid": trees = (best_iteration + 1) * ROUND_SCALE
                             # (some use (n_train + n_valid) / n_train since more data supports more trees)

MODEL_DIR = Path("models/xgb_final")          # can be a volume path; files are copied there whole

## 2. Dask (CUDA) cluster

In [ ]:
client = make_client(use_gpu=USE_GPU)
client

## 3. Selected hyper-parameters

In [ ]:
if PARAM_SOURCE == "study" and not (STUDY_DIR / "study.db").exists() and TRAIN_PATH is None:
    print(f"Demo: no study at {STUDY_DIR}, falling back to MANUAL_PARAMS")
    PARAM_SOURCE = "manual"

if PARAM_SOURCE == "study":
    tuned = load_tuned_params(STUDY_DIR, trial=TRIAL, max_overfit=MAX_OVERFIT)
    LOSS, params = tuned["loss"], tuned["params"]
    tuned_rounds = tuned["best_iteration"] + 1
    print(f"Study {STUDY_DIR} -> trial #{tuned['trial']} (loss={LOSS}, overfit index={tuned['overfit_index']:.4f})")
    display(pd.DataFrame({"train": tuned["train_metrics"], "valid": tuned["valid_metrics"]}).rename_axis("tuning metrics"))
else:
    tuned = None
    params, tuned_rounds = dict(MANUAL_PARAMS), MANUAL_NUM_ROUNDS
    if LOSS != "tweedie":
        params.pop("tweedie_variance_power", None)

loss = get_loss(LOSS)
print(f"loss = {LOSS} ({loss.objective}); trees in the tuning run = {tuned_rounds}")
pd.Series(params, name="value", dtype=object).to_frame()

## 4. Data

In [ ]:
if TRAIN_PATH is None:  # synthetic demo data (same as the tuning demo) + a test set
    import dask.dataframe as dd
    data_dir = Path("demo_data"); data_dir.mkdir(exist_ok=True)
    if not (data_dir / "train").exists():
        train_df, valid_df = make_synthetic_data(n_train=500_000, n_valid=100_000, n_features=20)
        dd.from_pandas(train_df, npartitions=16).to_parquet(data_dir / "train", overwrite=True)
        dd.from_pandas(valid_df, npartitions=8).to_parquet(data_dir / "valid", overwrite=True)
    if not (data_dir / "test").exists():
        test_df, _ = make_synthetic_data(n_train=100_000, n_valid=0, n_features=20, seed=1)
        dd.from_pandas(test_df, npartitions=8).to_parquet(data_dir / "test", overwrite=True)
    TRAIN_PATH, VALID_PATH, TEST_PATH = (str(data_dir / s) for s in ("train", "valid", "test"))

X_train, y_train, X_valid, y_valid = load_dask_data(
    TRAIN_PATH, VALID_PATH, TARGET, features=FEATURES, fmt=FILE_FORMAT, use_gpu=USE_GPU, client=client)
features = list(X_train.columns)
if TEST_PATH is not None:
    X_test, y_test = load_dask_split(
        TEST_PATH, TARGET, features=features, fmt=FILE_FORMAT, use_gpu=USE_GPU, client=client)
else:
    X_test = y_test = None
print(f"train {len(y_train):,} | valid {len(y_valid):,} | test {0 if y_test is None else len(y_test):,} rows; {len(features)} features")

## 5. Fit

* `early_stopping`: train on **train**, stop when the loss metric on **valid** has not improved for 50 rounds.
* `train_plus_valid`: train on **train + valid** with a fixed number of trees. The monitoring set (test if given,
  otherwise valid, which is then in-sample) only records metrics; it does not stop training.

In [ ]:
max_bin = params.get("max_bin", 256)

if FIT_MODE == "early_stopping":
    fit_X, fit_y = X_train, y_train
    mon_X, mon_y, mon_name = X_valid, y_valid, "valid"
    num_rounds, es_rounds = NUM_BOOST_ROUND, EARLY_STOPPING_ROUNDS
elif FIT_MODE == "train_plus_valid":
    import dask.dataframe as dd
    fit_X = dd.concat([X_train, X_valid]).persist()
    fit_y = dd.concat([y_train, y_valid]).persist()
    if X_test is not None:
        mon_X, mon_y, mon_name = X_test, y_test, "test"
    else:
        mon_X, mon_y, mon_name = X_valid, y_valid, "valid (in-sample!)"
    num_rounds, es_rounds = max(1, int(round(tuned_rounds * ROUND_SCALE))), None
else:
    raise ValueError(FIT_MODE)

dfit, dmon = build_dmatrices(client, fit_X, fit_y, mon_X, mon_y, max_bin=max_bin)
booster, result = train_xgb_model(
    client, dfit, dmon, params, loss,
    X_train=fit_X, y_train=fit_y, X_valid=mon_X, y_valid=mon_y,
    num_boost_round=num_rounds, early_stopping_rounds=es_rounds, use_gpu=USE_GPU,
)
print(f"{FIT_MODE}: {booster.num_boosted_rounds()} trees in {result.train_seconds:.1f}s; monitoring set = {mon_name}")
fit_table = pd.DataFrame({"fit data": result.train_metrics, mon_name: result.valid_metrics})
fit_table.loc["avg_target"] = [result.avg_target_train, result.avg_target_valid]
fit_table.loc["avg_prediction"] = [result.avg_pred_train, result.avg_pred_valid]
fit_table

In [ ]:
# Learning curves of the loss metric
import matplotlib.pyplot as plt

m = result.loss_metric
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(result.history["train"][m], label="fit data")
ax.plot(result.history["valid"][m], label=mon_name)
if FIT_MODE == "early_stopping":
    ax.axvline(result.best_iteration, ls="--", color="grey", label=f"best iteration {result.best_iteration}")
ax.set(xlabel="boosting round", ylabel=m, title="Learning curve")
ax.legend();

## 6. Evaluation

Metrics on each split, computed from the predictions on the driver (NumPy). `normalized_gini` measures how well the
model ranks rows (1 = perfect), which matters for a skewed target where a few large values dominate RMSE.

In [ ]:
huber_slope = params.get("huber_slope", 1.0)
tweedie_power = params.get("tweedie_variance_power") if LOSS == "tweedie" else None

splits = {"train": (X_train, y_train), "valid": (X_valid, y_valid)}
if X_test is not None:
    splits["test"] = (X_test, y_test)

preds, metrics = {}, {}
for name, (X, y) in splits.items():
    y_np, p_np = to_numpy(y), to_numpy(predict(client, booster, X))
    preds[name] = (y_np, p_np)
    metrics[name] = regression_metrics(y_np, p_np, huber_slope=huber_slope, tweedie_power=tweedie_power)
metrics_df = pd.DataFrame(metrics)
if FIT_MODE == "train_plus_valid":
    print("Note: valid is part of the training data in this mode; judge the model on test.")
metrics_df

### Decile table (calibration by predicted value)

Rows are split into 10 equal-size groups by prediction. Look for: average prediction close to average target in every
bin (`pred / target` ≈ 1), and a steep rise in `avg_target` from bin 1 to bin 10 (the model separates high values).

In [ ]:
eval_split = "test" if "test" in preds else "valid"
deciles = decile_table(*preds[eval_split], n_bins=10)
deciles.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(deciles.index, deciles["avg_target"], "o-", label="avg target")
axes[0].plot(deciles.index, deciles["avg_prediction"], "s--", label="avg prediction")
axes[0].set(xlabel="predicted decile", ylabel="value", title=f"Calibration by decile ({eval_split})")
axes[0].legend()

y_np, p_np = preds[eval_split]
lim = np.quantile(np.concatenate([y_np, p_np]), 0.999)
axes[1].scatter(p_np, y_np, s=2, alpha=0.2)
axes[1].plot([0, lim], [0, lim], color="C3", lw=1)
axes[1].set(xlim=(0, lim), ylim=(0, lim), xlabel="prediction", ylabel="target",
            title=f"Target vs prediction ({eval_split}, up to 99.9th pct)")
plt.tight_layout()

## 7. Feature importance

In [ ]:
imp = pd.DataFrame({
    kind: pd.Series(booster.get_score(importance_type=kind))
    for kind in ("total_gain", "gain", "weight", "cover")
}).fillna(0.0)
imp["total_gain_share"] = imp["total_gain"] / imp["total_gain"].sum()
imp = imp.sort_values("total_gain", ascending=False)

top = imp.head(20)[::-1]
fig, ax = plt.subplots(figsize=(7, 0.3 * len(top) + 1))
ax.barh(top.index, top["total_gain_share"])
ax.set(xlabel="share of total gain", title="Top features")
plt.tight_layout()
imp.head(20)

## 8. Save the model

`MODEL_DIR` gets `booster.ubj`, `metrics.json` (fit metrics + learning curves) and `model_card.json` (hyper-parameters,
source trial, fit mode, features, evaluation metrics). Files are written locally then copied, so a volume path works.

In [ ]:
model_card = {
    "loss": LOSS,
    "objective": loss.objective,
    "params": params,
    "param_source": PARAM_SOURCE,
    "study_dir": str(STUDY_DIR) if PARAM_SOURCE == "study" else None,
    "trial": tuned["trial"] if tuned else None,
    "fit_mode": FIT_MODE,
    "num_trees": booster.num_boosted_rounds(),
    "features": features,
    "target": TARGET,
    "data": {"train": TRAIN_PATH, "valid": VALID_PATH, "test": TEST_PATH},
    "evaluation": metrics,
}
save_model_bundle(booster, result, MODEL_DIR, model_card)
print("saved to", MODEL_DIR, "->", sorted(p.name for p in Path(MODEL_DIR).iterdir()))

## 9. Reload and score new data

In [ ]:
import xgboost as xgb

card = json.loads((MODEL_DIR / "model_card.json").read_text())
final_booster = xgb.Booster()
final_booster.load_model(MODEL_DIR / "booster.ubj")

# Score any Dask (cuDF) frame with the same feature columns, e.g. the validation set:
scores = predict(client, final_booster, X_valid[card["features"]])
check = float(scores.mean().compute())
print(f"avg score = {check:.4f}  (matches fitted model: {np.isclose(check, preds['valid'][1].mean(), rtol=1e-4)})")

In [ ]:
client.close()